# Scheme Navigator — free IndicTrans2 pack generator
Run this on a **Colab GPU runtime**. The cache is copied to Google Drive so disconnects are resumable.

**Before running:** request/accept access to `ai4bharat/indictrans2-en-indic-dist-200M` on Hugging Face, then create a **Read** access token from the same account. The token is entered into a hidden prompt and is never saved in this notebook/repository.

In Colab choose **Runtime → Change runtime type → T4 GPU** before running the translation cell. The notebook intentionally keeps Colab's preinstalled GPU-enabled PyTorch instead of upgrading it.


In [ ]:
import os
from google.colab import drive

# IMPORTANT: never delete the repo while Colab is still inside it.
os.chdir('/content')
drive.mount('/content/drive', force_remount=False)
!rm -rf /content/scheme-navigator
!git clone https://github.com/um26/scheme-navigator.git /content/scheme-navigator
os.chdir('/content/scheme-navigator')
print('✅ Working directory:', os.getcwd())


In [ ]:
import os
os.chdir('/content/scheme-navigator')

# Keep Colab's own CUDA-enabled torch. Upgrading torch alone can leave its
# preinstalled torchvision binary incompatible (the torchvision::nms error).
# IndicTrans2 is text-only, so torchvision is unnecessary here.
!pip -q uninstall -y torchvision >/dev/null 2>&1 || true
!pip -q install -U 'transformers>=4.51,<5' indictranstoolkit sentencepiece sacremoses accelerate huggingface_hub
!npm install --silent

import torch
from packaging.version import Version
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if Version(torch.__version__.split('+')[0]) < Version('2.5'):
    raise RuntimeError('IndicTransToolkit needs torch>=2.5. Start a fresh Colab runtime; do not manually install a CPU-only torch build.')
print('✅ Dependencies installed without replacing Colab PyTorch')


## Authenticate to Hugging Face
The web page saying **granted access** is necessary, but Colab must also log in with a token from that **same account**. Paste a Hugging Face **Read** token into the hidden prompt. Do not paste the token into chat, notebook text, screenshots, or GitHub.


In [ ]:
import os
from getpass import getpass
from huggingface_hub import HfApi

os.chdir('/content/scheme-navigator')
MODEL_NAME = 'ai4bharat/indictrans2-en-indic-dist-200M'
HF_TOKEN = getpass('Paste your Hugging Face READ token (input is hidden): ')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

api = HfApi(token=HF_TOKEN)
me = api.whoami()
print(f"✅ Authenticated to Hugging Face as: {me.get('name') or me.get('fullname') or 'your account'}")
try:
    info = api.model_info(MODEL_NAME, token=HF_TOKEN)
    print(f"✅ Gated model access verified: {info.modelId}")
except Exception as exc:
    raise RuntimeError(
        'Hugging Face login worked, but this token/account still cannot access the IndicTrans2 model. '
        'Make sure the token was created from the SAME account that shows granted access, then rerun this cell.'
    ) from exc


In [ ]:
import os
from pathlib import Path

os.chdir('/content/scheme-navigator')
assert Path('scripts/translate-catalog.py').exists(), 'Repo setup is incomplete. Rerun the first setup cell.'
# Restore resumable cache from Drive, if one exists.
!mkdir -p .translation-work public/i18n/schemes
!cp -r /content/drive/MyDrive/scheme-navigator-i18n/cache .translation-work/ 2>/dev/null || true
!cp /content/drive/MyDrive/scheme-navigator-i18n/generated-ui.json public/i18n/generated-ui.json 2>/dev/null || true
!cp /content/drive/MyDrive/scheme-navigator-i18n/generated.js lib/i18n/generated.js 2>/dev/null || true
!cp /content/drive/MyDrive/scheme-navigator-i18n/*.json.gz public/i18n/schemes/ 2>/dev/null || true
print('✅ Repo/cache ready at', os.getcwd())


In [ ]:
import os
from pathlib import Path
import torch

os.chdir('/content/scheme-navigator')
assert Path('scripts/translate-catalog.py').exists(), 'Translation script missing. Rerun the setup cell.'
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. In Colab choose Runtime → Change runtime type → T4 GPU, then restart and rerun from the top.')
print('✅ GPU:', torch.cuda.get_device_name(0))

# Smoke-test one language first. Once Hindi finishes and packages correctly,
# change this to another language or a small comma-separated batch.
LOCALES = 'hi'
!python /content/scheme-navigator/scripts/translate-catalog.py --locales $LOCALES --batch-size 32


In [ ]:
import os
from pathlib import Path

os.chdir('/content/scheme-navigator')
# Persist progress + create an uploadable artifact only after translation succeeds.
packs = list(Path('public/i18n/schemes').glob('*.json.gz'))
if not packs:
    raise RuntimeError('No translation packs exist yet. Run the translation cell successfully before packaging.')

!mkdir -p /content/drive/MyDrive/scheme-navigator-i18n
!rm -rf /content/drive/MyDrive/scheme-navigator-i18n/cache
!cp -r .translation-work/cache /content/drive/MyDrive/scheme-navigator-i18n/cache
!cp public/i18n/generated-ui.json /content/drive/MyDrive/scheme-navigator-i18n/generated-ui.json
!cp lib/i18n/generated.js /content/drive/MyDrive/scheme-navigator-i18n/generated.js
!cp public/i18n/schemes/*.json.gz /content/drive/MyDrive/scheme-navigator-i18n/
!rm -f /content/scheme-navigator-translations.zip
!zip -q -r /content/scheme-navigator-translations.zip public/i18n lib/i18n/generated.js
print('✅ Translation packs:', ', '.join(p.name for p in packs))
print('Artifact: /content/scheme-navigator-translations.zip')
